# 说明

代码原版使用了 Azure AI Search 服务，需要在 Azure 平台注册和配置。这里保留原教程的整体流程，但将检索和记忆存储层改写为本地 `ChromaDB`，这样不依赖 Azure AI Search 也可以学习和运行。

同时，本 notebook 使用：

- `qwen-max` 作为聊天模型，通过 DashScope 的 OpenAI 兼容接口访问
- 项目本地 `models` 目录中的 `BAAI/bge-small-en-v1.5` 作为 embedding 模型

这样你可以把“聊天模型”和“向量模型”分开理解：

- 聊天模型负责推理、对话、工具调用
- embedding 模型负责把文本转换成向量，供 ChromaDB 和 Mem0 做语义检索

注意：本 notebook 不再依赖远程 embedding 服务，但仍然需要 `DASHSCOPE_API_KEY` 来调用 `qwen-max`。


# 使用 Mem0、Semantic Kernel 和 ChromaDB 构建具有持久记忆的 AI 代理

本笔记本演示如何构建一个智能旅行预订代理，该代理能够在对话中记住用户偏好。通过结合 Mem0、Semantic Kernel 和 ChromaDB，我们创建了一个基于历史交互提供个性化旅行推荐的代理。

## 你将学到什么

1. **Mem0 记忆管理**：存储和检索用户偏好
2. **ChromaDB 作为本地向量存储**：使用语义搜索存储酒店数据和记忆
3. **Semantic Kernel 代理**：构建能够使用工具和记忆的智能代理
4. **本地 BGE embedding 模型**：使用项目里的本地向量模型完成文本嵌入
5. **跨会话持久化**：让代理在不同对话间记住用户信息

## 前置条件

- 已配置 DashScope API Key
- 本地可以安装并运行 `chromadb`
- 已安装 `semantic-kernel` 和 `mem0ai`
- 项目本地已存在 `BAAI/bge-small-en-v1.5` 模型文件
- 使用 OpenAI 兼容方式调用通义千问模型


## 理解内存架构

### 什么是 Mem0？

**Mem0** 是一个智能内存层，提供以下功能：
- **长期记忆**：存储用户偏好、过去的互动以及学习到的信息
- **语义搜索**：根据上下文检索相关记忆
- **用户特定存储**：为不同用户维护独立的内存空间
- **自动相关性**：根据当前上下文呈现最相关的记忆

### 各组件如何协同工作：
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Semantic       │────▶│      Mem0        │────▶│  Azure AI       │
│  Kernel Agent   │     │  Memory Layer    │     │  Search         │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                       │                         │
         │                       │                         │
    Processes              Stores/Retrieves          Vector Store
    User Input             User Preferences         for Memories &
                          & Context                  Travel Data
```


In [ ]:
# 安装本 notebook 需要的两个核心依赖。
! pip install mem0ai "chromadb~=0.6.3"


## 导入所需的包


In [34]:
# 导入 json，用于把 Python 数据结构转成 JSON 字符串输出。
import json
# 导入 os，用于读取环境变量。
import os
# 从 pathlib 导入 Path，用于更稳妥地处理本地文件路径。
from pathlib import Path
# 从 typing 中导入类型注解工具，方便给函数参数和返回值写说明。
from typing import Annotated, List, Dict, Any, TYPE_CHECKING
# 导入 datetime，用于生成带时间戳的测试用户。
from datetime import datetime
# 导入 uuid，保留给后续扩展唯一标识时使用。
import uuid

# 从 IPython.display 导入显示工具，用于在 notebook 里渲染富文本输出。
from IPython.display import display, HTML, Markdown
# 导入 load_dotenv，用于读取 .env 文件中的密钥和配置。
from dotenv import load_dotenv
# 导入 chromadb，用作本地向量数据库。
import chromadb

# 从 openai 导入 AsyncOpenAI，用于连接 DashScope 的 OpenAI 兼容接口。
from openai import AsyncOpenAI
# 从 mem0 导入 Memory，这是本教程使用的长期记忆层。
from mem0 import Memory

# 从 semantic_kernel 导入 Kernel，它是插件和 AI 服务的注册容器。
from semantic_kernel import Kernel
# 导入 ChatCompletionAgent，用于创建具备工具调用能力的聊天代理。
from semantic_kernel.agents import ChatCompletionAgent
# 导入 OpenAIChatCompletion，用于连接 OpenAI 兼容接口。
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
# 导入 FunctionChoiceBehavior，保留给后续扩展函数调用策略时使用。
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
# 导入 kernel_function 装饰器，用于把普通 Python 方法暴露成 AI 可调用工具。
from semantic_kernel.functions import kernel_function
# 导入 ChatHistory，保留给后续扩展对话历史结构时使用。
from semantic_kernel.contents import ChatHistory
# 再导入 ChatCompletionAgent 和 ChatHistoryAgentThread，其中 ChatHistoryAgentThread 用来维护线程级对话历史。
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread

# 只有在类型检查时才导入 Collection，避免运行时增加不必要依赖。
if TYPE_CHECKING:
    # 导入 Chroma 集合类型，仅用于类型提示，不影响实际执行。
    from chromadb.api.models.Collection import Collection


## 环境配置


In [35]:
# 从 .env 文件加载环境变量到当前 Python 进程。
load_dotenv()

# 指定要使用的通义千问模型名称。
model_name = "qwen-max"
# 读取 DashScope 的 API Key。
dashscope_api_key = os.getenv("DASHSCOPE_API_KEY")
# 指定 DashScope 的 OpenAI 兼容接口地址。
dashscope_base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"

# 定义本地 BGE 模型相对于项目根目录的位置。
embedding_model_relative_path = Path(
    "models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
# bge-small-en-v1.5 的向量维度是 384。
embedding_model_dims = 384

# 准备一组候选根目录，用来兼容 notebook 从不同目录启动的情况。
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
    Path("/Users/a1-6/Desktop/AIAgent"),
]

# 先把 embedding_model_name 置空，后面找到真实路径后再赋值。
embedding_model_name = None
# 遍历这些候选根目录。
for root in candidate_roots:
    # 拼出当前候选根目录下的模型绝对路径。
    candidate_path = (root / embedding_model_relative_path).resolve()
    # 如果这个路径真实存在，就说明找到了模型目录。
    if candidate_path.exists():
        # 保存找到的绝对路径字符串。
        embedding_model_name = str(candidate_path)
        # 打印找到的模型路径，便于调试。
        print(f"✅ Found local embedding model at: {embedding_model_name}")
        # 找到后立即退出循环。
        break

# 如果遍历完仍然没有找到模型目录，就主动抛错并给出清晰提示。
if embedding_model_name is None:
    raise FileNotFoundError(
        f"Local embedding model not found. Tried roots: {[str(root) for root in candidate_roots]}"
    )

# 定义存储酒店数据的 Chroma collection 名称。
travel_collection_name = "travel_hotels"
# 定义酒店向量库在本地磁盘上的持久化目录。
travel_chroma_path = "./chroma_travel_db"
# 定义 Mem0 使用的记忆 collection 名称。
memory_collection_name = "mem0"
# 定义记忆向量库在本地磁盘上的持久化目录。
memory_chroma_path = "./chroma_mem0_db"

# 创建 OpenAI 兼容客户端，底层实际连接的是 DashScope。
client = AsyncOpenAI(
    api_key=dashscope_api_key,
    base_url=dashscope_base_url,
)


✅ Found local embedding model at: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


## 初始化 ChromaDB 以处理旅游数据

首先，我们将使用示例酒店和目的地数据设置本地 `ChromaDB` 集合，供代理进行语义搜索。


In [36]:
# 创建一个持久化的 Chroma 客户端，用于存储酒店搜索数据。
travel_chroma_client = chromadb.PersistentClient(path=travel_chroma_path)
# 获取或创建酒店搜索集合，后续所有酒店检索都从这里进行。
travel_collection = travel_chroma_client.get_or_create_collection(
    # 指定集合名称，便于后续复用。
    name=travel_collection_name,
    # 写入一点集合级元数据，方便理解它的用途。
    metadata={"description": "travel hotel search collection"}
)

# 读取当前集合里已经存在的所有文档 ID。
existing_ids = travel_collection.get().get("ids", [])
# 如果这个集合里已经有旧数据，就先删除，避免重复插入演示数据。
if existing_ids:
    # 按 ID 批量删除已有文档，让每次跑 notebook 都得到一致结果。
    travel_collection.delete(ids=existing_ids)

# 打印提示，说明本地酒店向量库已经准备好了。
print(f"✅ ChromaDB collection '{travel_collection_name}' is ready")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given


✅ ChromaDB collection 'travel_hotels' is ready


In [37]:
# 定义示例酒店数据列表，后续会把它们写入本地 ChromaDB。
sample_hotels = [
    # 第 1 家酒店的结构化信息。
    {
        # 唯一 ID，用于写入和检索。
        "id": "1",
        # 酒店名称。
        "name": "Le Meurice Paris",
        # 酒店描述，包含卖点信息。
        "description": "Luxury palace hotel with Michelin-starred dining and views of the Tuileries Garden",
        # 所在地。
        "location": "Paris, France",
        # 配套设施。
        "amenities": "Spa, Michelin Restaurant, Concierge, Room Service, Fitness Center",
        # 每晚价格。
        "price_per_night": 850,
        # 评分。
        "rating": 4.8,
        # 语义标签，用于增强检索。
        "tags": ["luxury", "romantic", "historic", "fine-dining", "spa"]
    },
    # 第 2 家酒店。
    {
        # 唯一 ID。
        "id": "2",
        # 酒店名称。
        "name": "Four Seasons Maui",
        # 酒店描述。
        "description": "Beachfront resort with world-class spa and family-friendly activities",
        # 所在地。
        "location": "Maui, Hawaii",
        # 配套设施。
        "amenities": "Beach Access, Kids Club, Multiple Pools, Spa, Golf Course",
        # 每晚价格。
        "price_per_night": 695,
        # 评分。
        "rating": 4.7,
        # 标签。
        "tags": ["beach", "family-friendly", "resort", "spa", "golf"]
    },
    # 第 3 家酒店。
    {
        # 唯一 ID。
        "id": "3",
        # 酒店名称。
        "name": "Aman Tokyo",
        # 酒店描述。
        "description": "Minimalist luxury hotel with panoramic city views and traditional onsen",
        # 所在地。
        "location": "Tokyo, Japan",
        # 配套设施。
        "amenities": "Onsen, City Views, Fine Dining, Spa, Business Center",
        # 每晚价格。
        "price_per_night": 780,
        # 评分。
        "rating": 4.9,
        # 标签。
        "tags": ["luxury", "business", "spa", "city", "minimalist"]
    },
    # 第 4 家酒店。
    {
        # 唯一 ID。
        "id": "4",
        # 酒店名称。
        "name": "Hotel Sacher Vienna",
        # 酒店描述。
        "description": "Historic hotel home of the original Sachertorte with elegant rooms",
        # 所在地。
        "location": "Vienna, Austria",
        # 配套设施。
        "amenities": "Historic Cafe, Concierge, Accessible Rooms, Pet-Friendly",
        # 每晚价格。
        "price_per_night": 420,
        # 评分。
        "rating": 4.6,
        # 标签。
        "tags": ["historic", "accessible", "pet-friendly", "cultural", "cafe"]
    },
    # 第 5 家酒店。
    {
        # 唯一 ID。
        "id": "5",
        # 酒店名称。
        "name": "Fairmont Whistler",
        # 酒店描述。
        "description": "Ski-in/ski-out resort with family suites and mountain views",
        # 所在地。
        "location": "Whistler, Canada",
        # 配套设施。
        "amenities": "Ski Access, Family Suites, Heated Pool, Kids Programs",
        # 每晚价格。
        "price_per_night": 380,
        # 评分。
        "rating": 4.5,
        # 标签。
        "tags": ["ski", "family-friendly", "mountain", "resort", "accessible"]
    }
]

# 把示例酒店批量写入 Chroma collection。
travel_collection.add(
    # 提取每家酒店的 ID 列表，作为向量库主键。
    ids=[hotel["id"] for hotel in sample_hotels],
    # 把结构化酒店信息拼成自然语言文本，便于语义检索。
    documents=[
        # 这里把名称、描述、设施、位置、标签拼成一段检索文本。
        f'{hotel["name"]}. {hotel["description"]}. Amenities: {hotel["amenities"]}. '
        # 继续拼接位置和标签信息。
        f'Location: {hotel["location"]}. Tags: {", ".join(hotel["tags"])}.'
        # 对每一家酒店都生成一段检索文档。
        for hotel in sample_hotels
    ],
    # 同时保留结构化 metadata，后面返回结果时直接展示这些字段。
    metadatas=[
        # 为每家酒店构造对应的元数据字典。
        {
            # 保存酒店名称。
            "name": hotel["name"],
            # 保存位置。
            "location": hotel["location"],
            # 保存描述。
            "description": hotel["description"],
            # 保存设施。
            "amenities": hotel["amenities"],
            # 保存价格。
            "price_per_night": hotel["price_per_night"],
            # 保存评分。
            "rating": hotel["rating"],
            # 把标签列表转成字符串，便于展示。
            "tags": ", ".join(hotel["tags"]),
        }
        # 对每家酒店都生成一份 metadata。
        for hotel in sample_hotels
    ],
)

# 打印写入完成提示。
print(f"✅ Uploaded {len(sample_hotels)} hotels to ChromaDB")
# 再打印一条中文提示，便于学习时查看。
print(f"✅ 已上传 {len(sample_hotels)} 家酒店到本地 ChromaDB 集合")


✅ Uploaded 5 hotels to ChromaDB
✅ 已上传 5 家酒店到本地 ChromaDB 集合


## 使用 ChromaDB 初始化 Mem0

现在我们将配置 Mem0 使用本地 `ChromaDB` 作为其持久记忆的向量存储。


In [38]:
# 定义 Mem0 的整体配置字典。
mem0_config = {
    # 配置 Mem0 内部调用的 LLM，用于理解和抽取用户记忆。
    "llm": {
        # 这里改成 openai 提供方，因为 DashScope 走的是 OpenAI 兼容接口。
        "provider": "openai",
        # 下面是该提供方的详细配置。
        "config": {
            # 使用通义千问的 qwen-max 模型。
            "model": model_name,
            # 温度设低一点，让记忆抽取更稳定。
            "temperature": 0.2,
            # 限制返回最大 token 数。
            "max_tokens": 1500,
            # 传入 DashScope API Key。
            "api_key": dashscope_api_key,
            # 指定 OpenAI 兼容接口的 base URL。
            "openai_base_url": dashscope_base_url,
        }
    },
    # 配置 Mem0 底层使用的向量数据库。
    "vector_store": {
        # 指定向量库提供方为 chromadb。
        "provider": "chroma",
        # 下面是 ChromaDB 的具体配置。
        "config": {
            # 指定记忆集合名称。
            "collection_name": memory_collection_name,
            # 指定持久化目录。
            "path": memory_chroma_path
        }
    },
    # 配置生成向量时使用的 embedding 模型。
    "embedder": {
        # 改成 huggingface 提供方，直接加载本地模型目录。
        "provider": "huggingface",
        # 下面是 huggingface embedder 的详细配置。
        "config": {
            # 使用项目本地的 BAAI/bge-small-en-v1.5 模型快照目录。
            "model": embedding_model_name,
            # 指定 embedding 维度。
            "embedding_dims": embedding_model_dims,
            # 如需强制 CPU/禁用远程下载，可在这里继续加 model_kwargs。
            "model_kwargs": {
                "device": "cpu"
            }
        }
    }
}

# 根据配置创建 Mem0 记忆对象。
memory = Memory.from_config(mem0_config)
# 打印提示，说明接下来开始做一次最小化测试。
print("🧪 测试 Mem0 + ChromaDB + Qwen + Local BGE 设置...")

# 准备一组简单测试消息，模拟一轮用户和助手对话。
test_messages = [
    # 用户表达自己的偏好。
    {"role": "user", "content": "I prefer luxury hotels with spa services."},
    # 助手回复，表示已经记住偏好。
    {"role": "assistant", "content": "I'll remember you prefer luxury hotels with spa services for future recommendations."}
]

# 把这组测试消息写入记忆系统。
# 这里使用 infer=False，表示直接存储当前文本，不再让 Mem0 额外做事实抽取和实体链接。
memory.add(test_messages, user_id="test_user", metadata={"category": "preferences"}, infer=False)
# 读取 test_user 的所有记忆，验证写入是否成功。
test_memories = memory.get_all(filters={"user_id": "test_user"})

# 如果返回值是带 results 键的字典，就从 results 里取列表长度。
if isinstance(test_memories, dict) and "results" in test_memories:
    # 统计字典格式返回中的记忆条数。
    memory_count = len(test_memories.get("results", []))
# 否则直接把返回值当作列表计算长度。
else:
    # 统计列表格式返回中的记忆条数。
    memory_count = len(test_memories)

# 打印最终测试结果。
print(f"✅ Mem0 测试成功！找到 {memory_count} 条记忆")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🧪 测试 Mem0 + ChromaDB + Qwen + Local BGE 设置...
✅ Mem0 测试成功！找到 5 条记忆


## 创建旅行预订插件

此插件通过 `ChromaDB + Mem0` 提供搜索酒店和管理用户偏好的功能，其中：

- 聊天模型使用 `qwen-max`
- 向量嵌入使用项目本地的 `BAAI/bge-small-en-v1.5`


In [39]:
# 定义一个插件类，把“酒店搜索”和“用户记忆管理”封装成 AI 可调用工具。
class TravelBookingPlugin:
    # 这个类的文档字符串用于说明插件用途。
    """Plugin for searching hotels and managing user travel preferences
    这个插件让AI代理能够搜索酒店和管理用户偏好"""

    # 初始化方法接收酒店向量集合和 Mem0 记忆对象。
    def __init__(self, travel_collection: "Collection", memory: Memory):
        # 保存酒店集合，供搜索酒店时使用。
        self.travel_collection = travel_collection
        # 保存记忆对象，供存储和检索偏好时使用。
        self.memory = memory

    # 把这个方法注册成一个 AI 工具函数。
    @kernel_function(
        # 这段描述会告诉模型这个工具的能力。
        description="Search for hotels based on criteria like location, amenities, or tags"
    )
    # 定义酒店搜索函数。
    def search_hotels(
        # self 代表当前插件实例。
        self,
        # query 是用户的酒店搜索条件。
        query: Annotated[str, "Search query for hotels (location, amenities, etc.)"],
        # max_results 控制最多返回多少条结果，默认 3 条。
        max_results: Annotated[int, "Maximum number of results to return"] = 3
    ) -> Annotated[str, "List of hotels matching the search criteria"]:
        # 这个文档字符串用于解释函数用途。
        """Search for hotels in the travel database
        根据位置、设施或标签等条件搜索酒店"""
        # 在 ChromaDB 中执行语义检索。
        results = self.travel_collection.query(
            # 把用户查询包装成列表，符合 Chroma 的接口要求。
            query_texts=[query],
            # 指定最多返回几条候选结果。
            n_results=max_results,
            # 指定需要把原文、元数据和距离一起返回。
            include=["documents", "metadatas", "distances"]
        )

        # 准备一个空列表，用于存放整理后的酒店结果。
        hotels = []
        # 从查询结果中取出第一组 metadata 列表。
        metadatas = results.get("metadatas", [[]])[0]
        # 遍历每一条酒店 metadata。
        for metadata in metadatas:
            # 把每条 metadata 重新整理成更易读的字段结构。
            hotels.append({
                # 酒店名称。
                "name": metadata.get("name"),
                # 酒店位置。
                "location": metadata.get("location"),
                # 酒店描述。
                "description": metadata.get("description"),
                # 每晚价格。
                "price_per_night": metadata.get("price_per_night"),
                # 酒店评分。
                "rating": metadata.get("rating"),
                # 酒店设施。
                "amenities": metadata.get("amenities"),
                # 把逗号分隔的标签字符串重新拆成列表。
                "tags": metadata.get("tags", "").split(", ")
            })

        # 把酒店结果转成格式化 JSON 字符串返回给模型。
        return json.dumps(hotels, indent=2, ensure_ascii=False)

    # 把“存储用户偏好”也注册成一个工具函数。
    @kernel_function(
        # 描述这个工具的作用。
        description="Store user travel preferences and important information in memory"
    )
    # 定义写入用户偏好的函数。
    def store_user_preference(
        # self 代表当前插件实例。
        self,
        # user_id 用于区分不同用户的记忆空间。
        user_id: Annotated[str, "User identifier"],
        # preference 是要记住的偏好文本。
        preference: Annotated[str, "User preference or information to remember"]
    ) -> Annotated[str, "Confirmation of stored preference"]:
        # 文档字符串说明函数用途。
        """Store user preferences in Mem0 memory
        将用户偏好存储到Mem0记忆中"""
        # 打印调试信息，便于观察什么时候写入了记忆。
        print(f"DEBUG: Storing preference for {user_id}: {preference}")

        # 用 try 包裹，避免写入失败时 notebook 直接中断。
        try:
            # 把这条已经整理好的偏好直接写入 Mem0。
            # 这里使用 infer=False，避免 Mem0 再次做实体链接，从而绕开当前版本与 Chroma 的 metadata 兼容问题。
            self.memory.add(preference, user_id=user_id, infer=False)
            # 返回成功提示。
            return f"✅ 已存储: {preference}"
        # 如果写入时出错，就捕获异常。
        except Exception as e:
            # 返回错误信息，方便定位问题。
            return f"❌ 存储偏好时出错: {str(e)}"

    # 把“读取用户全部偏好”注册成工具函数。
    @kernel_function(
        # 描述这个工具函数的作用。
        description="Get all stored preferences for a user"
    )
    # 定义读取全部偏好的函数。
    def get_user_preferences(
        # self 代表当前插件实例。
        self,
        # 指定要读取哪个用户的记忆。
        user_id: Annotated[str, "User identifier"]
    ) -> Annotated[str, "All user preferences and memories"]:
        # 文档字符串说明函数用途。
        """Get all memories for a specific user
        获取特定用户的所有记忆"""
        # 打印调试信息。
        print(f"DEBUG: Getting all preferences for {user_id}")

        # 用 try 包裹读取过程。
        try:
            # 从 Mem0 中读取该用户的全部记忆。
            results = self.memory.get_all(filters={"user_id": user_id})

            # 如果返回值是字典格式，就取出真正的 results 列表。
            if isinstance(results, dict) and "results" in results:
                # 提取记忆列表。
                results = results.get("results", [])

            # 如果结果为空，就返回未找到提示。
            if not results:
                # 告诉调用方当前没有任何偏好。
                return f"未找到用户 {user_id} 的偏好"

            # 准备一个空列表，用于整理纯文本记忆。
            memories = []
            # 遍历所有记忆结果。
            for result in results:
                # 如果结果本身是字典结构。
                if isinstance(result, dict):
                    # 优先取其中的 memory 字段作为记忆正文。
                    memory_text = result.get("memory", str(result))
                    # 把记忆正文加入列表。
                    memories.append(memory_text)
                # 否则直接转成字符串处理。
                else:
                    # 把非字典结果也加入列表。
                    memories.append(str(result))

            # 把所有记忆拼成带项目符号的文本返回。
            return f"User preferences for {user_id}:\n- " + "\n- ".join(memories)

        # 如果读取过程报错，就捕获异常。
        except Exception as e:
            # 在 notebook 输出里打印详细错误。
            print(f"获取偏好时出错: {str(e)}")
            # 返回一个兜底提示。
            return f"No preferences found for user {user_id}"

    # 把“按语义搜索记忆”注册成工具函数。
    @kernel_function(
        # 描述这个工具函数的作用。
        description="Search user's memories for relevant information"
    )
    # 定义语义搜索用户记忆的函数。
    def search_memories(
        # self 代表当前插件实例。
        self,
        # 指定要搜索哪个用户的记忆。
        user_id: Annotated[str, "User identifier"],
        # 指定搜索问题，比如预算、饮食限制、地点偏好等。
        query: Annotated[str, "What to search for (e.g., 'family vacation', 'dietary restrictions')"]
    ) -> Annotated[str, "Relevant memories"]:
        # 文档字符串说明函数用途。
        """Search user memories using Mem0
        使用Mem0语义搜索用户记忆"""
        # 输出调试日志，方便查看具体搜索了什么。
        print(f"DEBUG: Searching memories for {user_id} with query: '{query}'")

        # 用 try 包裹搜索过程。
        try:
            # 调用 Mem0 的 search 做语义检索。
            results = self.memory.search(query, filters={"user_id": user_id})

            # 如果返回值是字典格式，就取出真正的结果列表。
            if isinstance(results, dict) and "results" in results:
                # 提取搜索命中的记忆列表。
                results = results.get("results", [])

            # 如果没有搜索到任何内容，就直接返回提示。
            if not results:
                # 告诉调用方本次查询没有命中记忆。
                return f"No memories found for query: {query}"

            # 准备一个列表，用于存放格式化后的记忆结果。
            memories = []
            # 遍历每一条记忆结果。
            for result in results:
                # 如果结果是字典格式。
                if isinstance(result, dict):
                    # 取出记忆正文。
                    memory_text = result.get("memory", str(result))
                    # 取出相关性分数。
                    score = result.get("score", None)
                    # 如果有分数，就一起展示出来。
                    if score:
                        # 将“记忆 + 分数”拼成一条结果。
                        memories.append(f"{memory_text} (relevance: {score:.2f})")
                    # 如果没有分数，就只展示记忆文本。
                    else:
                        # 直接加入纯文本记忆。
                        memories.append(memory_text)
                # 如果结果不是字典，就直接转成字符串。
                else:
                    # 把它加入结果列表。
                    memories.append(str(result))

            # 把结果拼成带项目符号的文本返回。
            return "Relevant memories:\n- " + "\n- ".join(memories)

        # 如果搜索过程发生异常。
        except Exception as e:
            # 打印错误详情。
            print(f"ERROR: {str(e)}")
            # 返回兜底结果，避免代理报错中断。
            return "No memories found."


## 初始化语义内核代理

使用旅行预订插件创建我们的旅行预订代理。


In [40]:
# 创建 Semantic Kernel 容器，用于注册 AI 服务和插件。
kernel = Kernel()

# 创建 OpenAI 兼容聊天服务对象，底层实际调用的是 DashScope 的 qwen-max。
chat_service = OpenAIChatCompletion(
    # 指定要使用的模型 ID。
    ai_model_id=model_name,
    # 传入前面创建的 AsyncOpenAI 客户端。
    async_client=client,
)
# 把聊天服务注册到 kernel 中。
kernel.add_service(chat_service)

# 根据酒店集合和记忆对象实例化旅行预订插件。
travel_plugin = TravelBookingPlugin(travel_collection, memory)
# 把插件注册到 kernel 中。
kernel.add_plugin(
    # 指定插件名。
    plugin_name="TravelBooking",
    # 传入插件实例。
    plugin=travel_plugin
)

# 创建带记忆能力的旅行助手代理。
travel_agent = ChatCompletionAgent(
    # 指定它使用哪个聊天服务。
    service=chat_service,
    # 给代理起一个名字。
    name="TravelBookingAssistant",
    # 用系统指令定义代理的工作流程和约束。
    # """
    #     你是一个有记忆的个性化旅行预订助理。
    #     工作流程:
    #     1. 当用户请求帮助时，使用search_memories（）和相关查询来搜索他们的记忆
    #     2. 利用这些回忆来个性化你的回应
    #     3. 使用store_user_preference（）存储它们提到的任何新首选项
    #     4. 当用户预订新的旅行时，首先通过创建关于酒店、饮食限制、位置、设施和预算的查询来检索用户的一般旅行偏好。然后使用search_hotels（）来查找合适的选项。
    #     5. 不要推荐超出预算的酒店。

    #     重要提示：对于所有内存操作（search_memories和store_user_preference），
    #     你必须使用user_id='sarah_johnson_123'。

    #     示例查询:
    #     -用户询问预订旅行→search_memories（query="preferences"）
    #     -用户询问预订旅行→search_memories（查询=“饮食限制”）
    #     -用户询问预订行程→search_memories（query="location"）
    #     -用户询问预订行程→search_memories（query=“便利设施”）
    #     -用户询问预订行程→search_memories（query="budget"）

    #     在回应时，一定要承认你在他们的记忆中发现了什么。
    # """

    instructions="""
    You are a personalized travel booking assistant with memory.

    WORKFLOW:
    1. When a user asks for help, search their memories using search_memories() with a relevant query
    2. Use the memories to personalize your response
    3. Store any new preferences they mention using store_user_preference()
    4. When the users is booking a new trip, first retrieve the users general travel preferences of the user by creating queries for hotels, dietary restrictions, location, amenities and budget. THEN use search_hotels() to find suitable options.
    5. Do not recommend hotels that are over budget.

    IMPORTANT: For ALL memory operations (search_memories and store_user_preference),
    you MUST use user_id='sarah_johnson_123' exactly as written.

    Example queries:
    - User asks about booking a trip → search_memories(query="preferences")
    - User asks about booking a trip → search_memories(query="dietary restrictions")
    - User asks about booking a trip → search_memories(query="location")
    - User asks about booking a trip → search_memories(query="amenities")
    - User asks about booking a trip → search_memories(query="budget")

    Always acknowledge what you found in their memories when responding.""",
    # 把插件暴露给代理，让模型能调用这些工具。
    plugins=[travel_plugin]
)


## 用于清晰显示的辅助函数


In [41]:
# 定义一个显示普通对话消息的辅助函数。
def display_message(role: str, content: str, color: str = "#2E8B57", emoji: str = ""):
    # 函数字符串说明其作用。
    """Display a message with nice formatting"""
    # 使用 f-string 动态拼出一段 HTML。
    html = f"""
    <div style='
        margin: 10px 0; 
        padding: 15px 20px; 
        border-left: 4px solid {color}; 
        background: rgba(128, 128, 128, 0.05); 
        border-radius: 8px;
    '>
        <strong style='color: {color}; font-size: 16px;'>{emoji} {role}:</strong><br>
        <div style='margin-top: 10px; white-space: pre-wrap; font-size: 14px; line-height: 1.6;'>{content}</div>
    </div>
    """
    # 把 HTML 包装后渲染到 notebook 输出中。
    display(HTML(html))

# 定义一个显示“记忆操作”的辅助函数。
def display_memory_operation(operation: str, details: str, color: str = "#9370DB"):
    # 函数字符串说明其作用。
    """Display memory operations for educational purposes"""
    # 构造一段用于展示记忆读写行为的 HTML。
    html = f"""
    <div style='
        margin: 5px 20px;
        padding: 10px 15px;
        background: rgba(147, 112, 219, 0.1);
        border: 1px solid {color};
        border-radius: 6px;
        font-family: monospace;
        font-size: 13px;
    '>
        <strong style='color: {color};'>🧠 Memory {operation}:</strong>
        <div style='margin-top: 5px; color: #555;'>{details}</div>
    </div>
    """
    # 渲染这段 HTML。
    display(HTML(html))

# 定义一个显示函数调用详情的辅助函数。
def display_function_call(function_name: str, args: dict, result: str = None):
    # 函数字符串说明其作用。
    """Display function calls for transparency"""
    # 先构造 details 折叠块的开头 HTML。
    html = f"""
    <details style='margin: 5px 20px; padding: 10px; background: rgba(0, 123, 255, 0.05); border: 1px solid #007BFF; border-radius: 6px;'>
        <summary style='cursor: pointer; font-weight: bold; color: #007BFF;'>⚙️ Function Call: {function_name}</summary>
        <div style='margin-top: 10px; font-family: monospace; font-size: 12px;'>
            <div><strong>Arguments:</strong> {json.dumps(args, indent=2)}</div>
    """
    # 如果函数有返回结果，就继续把结果 HTML 拼接进去。
    if result:
        # 把结果展示在一个 pre 块中，方便阅读。
        html += f"<div style='margin-top: 10px;'><strong>Result:</strong><pre style='background: #f8f8f8; padding: 8px; border-radius: 4px; overflow-x: auto;'>{result}</pre></div>"
    # 补上 HTML 的收尾标签。
    html += "</div></details>"
    # 最终把函数调用详情渲染出来。
    display(HTML(html))


## 演示带记忆功能的旅行预订

让我们通过真实的旅行预订场景来展示代理如何记住并使用用户偏好。


### 场景 1：首次使用者 - 周年旅行计划


In [42]:
# 定义演示用户的固定 user_id，真实系统通常来自登录态或数据库。
sarah_user_id = "sarah_johnson_123"

# 打印场景标题。
print("🎯 SCENARIO 1: Sarah's First Booking - Anniversary Trip\n")

# 创建一个新的对话线程对象，用于保存 Sarah 当前会话的上下文。
sarah_chat = ChatHistoryAgentThread()

# 构造 Sarah 第一轮消息，描述她的旅行需求和预算。
# 嗨！我是莎拉，我正在计划一次特别的旅行来庆祝我的十周年结婚纪念日。
# 我们喜欢浪漫的目的地，美食和水疗体验。我丈夫行动不便，
# 所以我们需要方便的住宿。我们的预算是每晚700-800美元
user_message1 = """Hi! I'm Sarah and I'm planning a special trip for my 10th wedding anniversary. 
We love romantic destinations, fine dining, and spa experiences. My husband has mobility issues, 
so we need accessible accommodations. Our budget is around $700-800 per night."""

# 在 notebook 中显示 Sarah 发出的第一条消息。
display_message("Sarah", user_message1, "#4fc3f7", "👤")

# 准备一个变量，用于保存代理最终回复文本。
response_content = ""
# 准备一个列表，用于去重记录已经展示过的函数调用。
function_calls_made = []

# 异步调用 travel_agent，让代理处理 Sarah 的需求。
async for response in travel_agent.invoke(
    # 把用户消息传给代理。
    messages=user_message1,
    # 把当前线程对象一并传入，这样代理能维护上下文。
    thread=sarah_chat
):
    # 如果本次流式返回里有内容片段。
    if response.content:
        # 就把它保存成当前最新的回复文本。
        response_content = str(response.content)

    # 如果响应对象里带有 thread 属性。
    if hasattr(response, 'thread'):
        # 取出最新线程对象，便于继续读消息历史。
        sarah_thread = response.thread

        # 遍历线程中的消息，查找函数调用记录。
        async for msg in sarah_thread.get_messages():
            # 只有当消息里存在 items 时才继续处理。
            if hasattr(msg, 'items') and msg.items:
                # 遍历这条消息中的每个 item。
                for item in msg.items:
                    # 只有 function_invoke_attempt 存在时，说明这里发生过工具调用。
                    if hasattr(item, 'function_invoke_attempt') and item.function_invoke_attempt:
                        # 取出函数调用对象。
                        func_call = item.function_invoke_attempt
                        # 把本次函数调用整理成一个结构化字典。
                        function_info = {
                            # 记录函数名。
                            'name': func_call.function_name,
                            # 记录函数参数。
                            'arguments': func_call.arguments,
                            # 如果有结果则取结果，否则记为 None。
                            'result': item.function_result.value if hasattr(item, 'function_result') else None
                        }
                        # 如果这条调用还没被展示过。
                        if function_info not in function_calls_made:
                            # 就先记录下来，防止重复展示。
                            function_calls_made.append(function_info)

                            # 如果调用的是“读取用户偏好”。
                            if 'get_user_preferences' in func_call.function_name:
                                # 就显示一次“记忆检索”提示。
                                display_memory_operation(
                                    "Retrieval", f"Checking existing preferences for user: {func_call.arguments.get('user_id', sarah_user_id)}")
                            # 如果调用的是“存储用户偏好”。
                            elif 'store_user_preference' in func_call.function_name:
                                # 就显示一次“记忆写入”提示。
                                display_memory_operation(
                                    "Storage", f"Storing: {func_call.arguments.get('preference', '')}")
                            # 如果调用的是“搜索酒店”。
                            elif 'search_hotels' in func_call.function_name:
                                # 就把函数名、参数和结果完整展示出来。
                                display_function_call(
                                    func_call.function_name,
                                    func_call.arguments,
                                    item.function_result.value if hasattr(
                                        item, 'function_result') else None
                                )

# 最后把代理给 Sarah 的自然语言回复显示出来。
display_message("Travel Assistant", response_content, "#81c784", "🤖")


🎯 SCENARIO 1: Sarah's First Booking - Anniversary Trip



DEBUG: Storing preference for sarah_johnson_123: romantic destinations, fine dining, and spa experiences for 10th wedding anniversary
DEBUG: Storing preference for sarah_johnson_123: accessible accommodations due to husband's mobility issues
DEBUG: Storing preference for sarah_johnson_123: budget is around $700-800 per night for the trip


In [43]:
# 构造第二轮用户消息，补充饮食偏好和过敏信息。
user_message2 = """The Hotel Sacher sounds perfect! We're both vegetarian and I have a severe nut allergy. 
Can you tell me more about their dining options?"""

# 在 notebook 中显示 Sarah 的第二条消息。
display_message("Sarah", user_message2, "#4fc3f7", "👤")

# 准备一个变量，用于保存第二轮回复内容。
response2_content = ""
# 异步调用代理继续处理第二轮对话。
async for response in travel_agent.invoke(
    # 把第二轮消息传给代理。
    messages=user_message2,
    # 延续上一轮的线程对象，让代理记住上下文。
    thread=sarah_thread
):
    # 如果当前流式响应片段有内容。
    if response.content:
        # 就更新当前回复文本。
        response2_content = str(response.content)

    # 如果响应对象里带有线程信息。
    if hasattr(response, 'thread'):
        # 更新线程对象。
        sarah_thread = response.thread

        # 遍历线程消息，找出新增的函数调用。
        async for msg in sarah_thread.get_messages():
            # 只有消息里有 items 才继续处理。
            if hasattr(msg, 'items') and msg.items:
                # 遍历每个 item。
                for item in msg.items:
                    # 只处理真实发生过的函数调用。
                    if hasattr(item, 'function_invoke_attempt') and item.function_invoke_attempt:
                        # 取出函数调用对象。
                        func_call = item.function_invoke_attempt
                        # 如果调用的是“存储用户偏好”，并且参数里确实带了 preference。
                        if 'store_user_preference' in func_call.function_name and func_call.arguments.get('preference'):
                            # 取出要存储的偏好文本。
                            pref = func_call.arguments.get('preference', '')
                            # 只把素食或坚果过敏相关的偏好专门展示出来。
                            if 'vegetarian' in pref.lower() or 'nut allergy' in pref.lower():
                                # 显示这次记忆存储操作。
                                display_memory_operation(
                                    "Storage", f"Storing: {pref}")

# 把第二轮代理回复显示出来。
display_message("Travel Assistant", response2_content, "#81c784", "🤖")


DEBUG: Storing preference for sarah_johnson_123: vegetarian diet for both, and a severe nut allergy to be considered for dining options


Number of requested results 80 is greater than number of elements in index 17, updating n_results = 17
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


DEBUG: Searching memories for sarah_johnson_123 with query: 'Hotel Sacher'


In [44]:
# 打印标题，准备直接检查本地 ChromaDB 中的记忆数据。
print("\n\n🔍 VERIFYING MEM0 STORAGE IN CHROMADB\n")

# 直接复用 Mem0 已经创建好的底层 collection，避免同一路径下重复初始化 Chroma client。
mem0_collection = memory.vector_store.collection

# 用 try 包裹检查逻辑，避免某一步出错导致 notebook 中断。
try:
    # 统计当前记忆集合中的总文档数。
    total_docs = mem0_collection.count()
    # 打印文档总数。
    print(f"📊 Total documents in Mem0 Chroma collection: {total_docs}")

    # 读取文档正文和 metadata，准备抽样展示。
    sample_docs = mem0_collection.get(include=["documents", "metadatas"])
    # 打印说明文字。
    print("\nSample documents:")
    # 同时遍历 id、正文和 metadata。
    for i, (doc_id, doc, meta) in enumerate(zip(
        # 取出所有 ID。
        sample_docs.get("ids", []),
        # 取出所有文档正文。
        sample_docs.get("documents", []),
        # 取出所有 metadata。
        sample_docs.get("metadatas", [])
    )):
        # 只展示前 3 条样例。
        if i < 3:
            # 打印第几条文档。
            print(f"\nDocument {i+1}:")
            # 打印文档 ID。
            print(f"  ID: {doc_id}")
            # 打印 metadata。
            print(f"  Metadata: {meta}")
            # 打印记忆正文。
            print(f"  Memory: {doc}")
# 如果检查过程中出错。
except Exception as e:
    # 打印错误信息。
    print(f"❌ Error checking Mem0 Chroma collection: {str(e)}")


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given




🔍 VERIFYING MEM0 STORAGE IN CHROMADB

📊 Total documents in Mem0 Chroma collection: 17

Sample documents:

Document 1:
  ID: 7914f379-43ad-4cfe-946b-31d3a7438163
  Metadata: {'attributed_to': 'user', 'category': 'preferences', 'created_at': '2026-05-03T15:01:09.236459+00:00', 'data': 'User prefers luxury hotels with spa services for future travel recommendations', 'hash': '0d6d1a7ce4bdf0537f690035b086611c', 'text_lemmatized': 'user prefer luxury hotel spa service future travel recommendation', 'updated_at': '2026-05-03T15:01:09.236459+00:00', 'user_id': 'test_user'}
  Memory: None

Document 2:
  ID: 00b2132b-c338-4ebc-99e0-5342cb16c96e
  Metadata: {'attributed_to': 'user', 'created_at': '2026-05-03T15:40:32.212646+00:00', 'data': 'User is planning a 10th wedding anniversary trip with a focus on romantic destinations, fine dining, and spa experiences.', 'hash': '021d0ad9dd358cb5582cd5dcf2daae9a', 'text_lemmatized': 'user plan planning 10th wedding anniversary trip focus romantic destin

In [45]:
# 打印标题，开始做更细的 Mem0 调试验证。
print("🔍 ENHANCED MEM0 VERIFICATION\n")

# 生成一个带时间戳的测试用户 ID，避免和之前数据冲突。
test_user = f"debug_user_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# 构造一条包含饮食和偏好的测试记忆。
test_memory = "I am vegetarian with a peanut allergy and love beach destinations"

# 打印第 1 步提示。
print(f"1. Adding memory for {test_user}...")
# 向 Mem0 写入测试记忆。
# 这里也使用 infer=False，便于测试基础存取链路是否正常。
add_result = memory.add(test_memory, user_id=test_user, infer=False)
# 打印原始返回结果，方便观察 Mem0 返回结构。
print(f"   Raw add result: {add_result}")

# 如果 add 返回的是带 results 字段的字典。
if isinstance(add_result, dict) and "results" in add_result:
    # 取出真正的结果列表。
    actual_results = add_result.get("results", [])
    # 打印本次实际新增的记忆条数。
    print(f"   Actual memories added: {len(actual_results) if isinstance(actual_results, list) else 0}")
    # 如果 results 本身是列表。
    if isinstance(actual_results, list):
        # 遍历每条新增记忆。
        for mem in actual_results:
            # 打印每条记忆的 ID 和正文。
            print(f"   - ID: {mem.get('id', 'N/A')}, Memory: {mem.get('memory', 'N/A')}")

# 打印第 2 步标题。
print("\n2. Testing get_all()...")
# 读取这个测试用户的所有记忆。
all_mems = memory.get_all(filters={"user_id": test_user})
# 打印原始返回值。
print(f"   Raw response: {all_mems}")
# 打印返回值类型。
print(f"   Response type: {type(all_mems)}")

# 如果返回值是字典结构。
if isinstance(all_mems, dict):
    # 打印它的所有键。
    print(f"   Dict keys: {list(all_mems.keys())}")
    # 如果里面包含 results。
    if "results" in all_mems:
        # 取出 results。
        results_value = all_mems["results"]
        # 打印 results 的类型。
        print(f"   'results' value type: {type(results_value)}")
        # 打印 results 的实际内容。
        print(f"   'results' value: {results_value}")

        # 如果 results 是列表。
        if isinstance(results_value, list):
            # 打印列表长度。
            print(f"   Number of memories: {len(results_value)}")
            # 遍历列表中的每条记忆。
            for i, mem in enumerate(results_value):
                # 逐条打印。
                print(f"   Memory {i}: {mem}")

# 打印第 3 步标题。
print("\n3. Testing search()...")
# 用“peanut allergy”做一次语义搜索。
search_results = memory.search("peanut allergy", filters={"user_id": test_user})
# 打印原始返回值。
print(f"   Raw response: {search_results}")
# 打印返回值类型。
print(f"   Response type: {type(search_results)}")

# 如果搜索返回的是字典。
if isinstance(search_results, dict):
    # 打印它的所有键。
    print(f"   Dict keys: {list(search_results.keys())}")
    # 如果里面有 results。
    if "results" in search_results:
        # 取出结果列表。
        results_value = search_results["results"]
        # 打印 results 的类型。
        print(f"   'results' value type: {type(results_value)}")
        # 打印 results 的具体内容。
        print(f"   'results' value: {results_value}")

# 打印第 4 步标题。
print("\n4. Testing direct ChromaDB access...")
# 用 try 包裹直接访问底层向量库的逻辑。
try:
    # 直接复用 Mem0 已经创建好的底层 collection。
    mem0_collection = memory.vector_store.collection
    # 直接从底层集合读取原始文档和 metadata。
    raw_docs = mem0_collection.get(include=["documents", "metadatas"])

    # 准备一个列表，用于收集当前测试用户的底层文档。
    matched = []
    # 同时遍历 ID、文档正文和 metadata。
    for doc_id, doc, meta in zip(
        # 读取所有 ID。
        raw_docs.get("ids", []),
        # 读取所有文档正文。
        raw_docs.get("documents", []),
        # 读取所有 metadata。
        raw_docs.get("metadatas", [])
    ):
        # 如果 metadata 是字典，且它的 user_id 等于当前测试用户。
        if isinstance(meta, dict) and meta.get("user_id") == test_user:
            # 就把这条底层文档保存下来。
            matched.append((doc_id, doc, meta))

    # 打印该测试用户在底层 Chroma 中命中的文档条数。
    print(f"   Documents found in Chroma for {test_user}: {len(matched)}")
    # 最多打印前 5 条匹配结果。
    for doc_id, doc, meta in matched[:5]:
        # 打印文档 ID。
        print(f"   - ID: {doc_id}")
        # 打印 metadata。
        print(f"     Metadata: {meta}")
        # 打印文档正文。
        print(f"     Memory: {doc}")

# 如果底层访问时出错。
except Exception as e:
    # 打印错误信息。
    print(f"   Error: {e}")

# 打印第 5 步标题。
print("\n5. Testing Mem0 version...")
# 如果 memory 对象暴露了 __version__ 属性。
if hasattr(memory, "__version__"):
    # 就打印它。
    print(f"   Mem0 version: {memory.__version__}")
# 如果 memory 对象暴露了 version 属性。
if hasattr(memory, "version"):
    # 也把它打印出来。
    print(f"   Mem0 version: {memory.version}")

# 打印第 6 步标题。
print("\n6. Available memory methods:")
# 遍历 memory 对象上的所有属性名。
for attr in dir(memory):
    # 只保留公开且可调用的方法。
    if not attr.startswith("_") and callable(getattr(memory, attr)):
        # 把方法名打印出来，方便学习 Mem0 API。
        print(f"   - {attr}")


Number of requested results 80 is greater than number of elements in index 18, updating n_results = 18


🔍 ENHANCED MEM0 VERIFICATION

1. Adding memory for debug_user_20260504_000736...
   Raw add result: {'results': [{'id': 'bcf21493-94c2-4147-8526-7549486f414f', 'memory': 'I am vegetarian with a peanut allergy and love beach destinations', 'event': 'ADD', 'actor_id': None, 'role': 'user'}]}
   Actual memories added: 1
   - ID: bcf21493-94c2-4147-8526-7549486f414f, Memory: I am vegetarian with a peanut allergy and love beach destinations

2. Testing get_all()...
   Raw response: {'results': [{'id': 'bcf21493-94c2-4147-8526-7549486f414f', 'memory': 'I am vegetarian with a peanut allergy and love beach destinations', 'hash': '361049df0e5c593f65ef695655e651ac', 'metadata': None, 'created_at': '2026-05-03T16:07:36.667408+00:00', 'updated_at': '2026-05-03T16:07:36.667408+00:00', 'user_id': 'debug_user_20260504_000736', 'role': 'user'}]}
   Response type: <class 'dict'>
   Dict keys: ['results']
   'results' value type: <class 'list'>
   'results' value: [{'id': 'bcf21493-94c2-4147-8526-754948

### 情景 2：回访 - 家庭度假（数周后）


In [46]:
# 打印场景 2 的标题，表示用户在几周后回来继续对话。
print("\n\n🎯 SCENARIO 2: Sarah Returns Weeks Later for Family Vacation\n")
# 打印说明，强调这是一个新的会话线程。
print("📅 Simulating time passing... Sarah starts a new conversation\n")

# 创建一个新的线程对象，模拟跨会话场景。
sarah_thread_new = ChatHistoryAgentThread()

# 构造 Sarah 回来后的新需求。
user_message3 = "Hi, my husband and I are planning another trip. We are looking for a good hotel!"

# 在 notebook 中显示 Sarah 的消息。
display_message("Sarah", user_message3, "#4fc3f7", "👤")

# 准备一个变量存放代理回复。
response3_content = ""
# 准备一个列表，保留给可能的记忆提取展示使用。
memories_retrieved = []

# 异步调用代理处理新会话中的请求。
async for response in travel_agent.invoke(
    # 传入用户消息。
    messages=user_message3,
    # 传入新的线程对象。
    thread=sarah_thread_new
):
    # 如果流式响应里带有文本片段。
    if response.content:
        # 就更新当前回复文本。
        response3_content = str(response.content)

    # 如果返回对象里包含 thread。
    if hasattr(response, 'thread'):
        # 更新线程对象。
        sarah_thread_new = response.thread

        # 遍历线程中的消息。
        async for msg in sarah_thread_new.get_messages():
            # 只处理带 items 的消息。
            if hasattr(msg, 'items') and msg.items:
                # 遍历消息中的每个 item。
                for item in msg.items:
                    # 只处理实际发生过的函数调用。
                    if hasattr(item, 'function_invoke_attempt') and item.function_invoke_attempt:
                        # 取出函数调用对象。
                        func_call = item.function_invoke_attempt

                        # 如果调用的是“读取全部偏好”，并且当前 item 带有函数结果。
                        if 'get_user_preferences' in func_call.function_name and hasattr(item, 'function_result'):
                            # 取出函数返回结果。
                            result = item.function_result.value
                            # 如果结果非空并且包含“User preferences”字样。
                            if result and "User preferences" in result:
                                # 展示这次记忆召回，让学习者看到跨会话记忆生效了。
                                display_memory_operation(
                                    "Retrieval", f"Found memories for {sarah_user_id}:\n{result}")

                        # 如果调用的是“存储用户偏好”。
                        elif 'store_user_preference' in func_call.function_name:
                            # 展示这次新记忆写入。
                            display_memory_operation(
                                "Storage", f"Storing: {func_call.arguments.get('preference', '')}")

                        # 如果调用的是“搜索酒店”。
                        elif 'search_hotels' in func_call.function_name:
                            # 把本次酒店搜索的函数调用细节展示出来。
                            display_function_call(
                                func_call.function_name,
                                func_call.arguments,
                                item.function_result.value if hasattr(
                                    item, 'function_result') else None
                            )

# 把代理给 Sarah 的回复展示出来。
display_message("Travel Assistant", response3_content, "#81c784", "🤖")




🎯 SCENARIO 2: Sarah Returns Weeks Later for Family Vacation

📅 Simulating time passing... Sarah starts a new conversation



Number of requested results 80 is greater than number of elements in index 18, updating n_results = 18
Number of requested results 80 is greater than number of elements in index 18, updating n_results = 18
Number of requested results 80 is greater than number of elements in index 18, updating n_results = 18
Number of requested results 80 is greater than number of elements in index 18, updating n_results = 18
Number of requested results 80 is greater than number of elements in index 18, updating n_results = 18


DEBUG: Searching memories for sarah_johnson_123 with query: 'preferences'
DEBUG: Searching memories for sarah_johnson_123 with query: 'dietary restrictions'
DEBUG: Searching memories for sarah_johnson_123 with query: 'location'
DEBUG: Searching memories for sarah_johnson_123 with query: 'amenities'
DEBUG: Searching memories for sarah_johnson_123 with query: 'budget'


In [47]:
# 构造 Sarah 的追问，让代理继续基于上下文回答。
user_message4 = "Great suggestions! For the Maui option, what activities would you recommend for the kids?"

# 在 notebook 中显示这条追问。
display_message("Sarah", user_message4, "#4fc3f7", "👤")

# 准备一个变量保存追问的回复内容。
response4_content = ""
# 继续在同一个新线程里调用代理。
async for response in travel_agent.invoke(
    # 传入追问内容。
    messages=user_message4,
    # 继续沿用新线程，保持上下文连续。
    thread=sarah_thread_new
):
    # 如果流式响应有内容。
    if response.content:
        # 就更新最终回复文本。
        response4_content = str(response.content)
    # 把返回的最新线程对象重新保存下来。
    sarah_thread_new = response.thread

# 把代理对追问的回答展示出来。
display_message("Travel Assistant", response4_content, "#81c784", "🤖")


In [48]:
# 打印标题，开始专门测试“用户偏好写入和读取”。
print("\n🧪 TESTING MEMORY RETRIEVAL\n")

# 构造一条新的测试偏好。
test_preference = "I love romantic destinations with spa services"
# 通过插件写入 Sarah 的偏好。
result = travel_plugin.store_user_preference(sarah_user_id, test_preference)
# 打印写入结果。
print(f"Store result: {result}")

# 通过插件读取 Sarah 的全部偏好。
preferences = travel_plugin.get_user_preferences(sarah_user_id)
# 打印读取结果。
print(f"\nRetrieved preferences:\n{preferences}")

# 直接通过底层 memory 对象读取 Sarah 的全部记忆。
direct_memories = memory.get_all(filters={"user_id": sarah_user_id})
# 打印直接读取的结果数量。
print(f"\nDirect memory.get_all() returned {len(direct_memories)} memories")
# 遍历所有直接读取到的记忆。
for i, mem in enumerate(direct_memories):
    # 按索引打印每一条记忆。
    print(f"Memory {i}: {mem}")



🧪 TESTING MEMORY RETRIEVAL

DEBUG: Storing preference for sarah_johnson_123: I love romantic destinations with spa services
Store result: ✅ 已存储: I love romantic destinations with spa services
DEBUG: Getting all preferences for sarah_johnson_123

Retrieved preferences:
User preferences for sarah_johnson_123:
- User is planning a 10th wedding anniversary trip with a focus on romantic destinations, fine dining, and spa experiences.
- User is looking for accessible accommodations for their 10th wedding anniversary trip.
- User has a budget of $700-800 per night for their 10th wedding anniversary trip.
- User's husband has mobility issues, which is why they need accessible accommodations for their 10th wedding anniversary trip
- User is a vegetarian and has a severe nut allergy.
- romantic destinations, fine dining, and spa experiences for 10th wedding anniversary
- need accessible accommodations due to husband's mobility issues
- budget is around $700-800 per night for the trip
- romantic

In [49]:
# 打印标题，准备直接检查底层 Chroma 集合的结构和内容。
print("\n🔍 CHECKING MEM0 COLLECTION STRUCTURE\n")

# 用 try 包裹，避免底层集合读取异常导致 notebook 终止。
try:
    # 直接复用 Mem0 已经创建好的底层 collection。
    mem0_collection = memory.vector_store.collection

    # 打印集合名称。
    print(f"Collection name: {memory_collection_name}")
    # 打印集合中的总文档数。
    print(f"Total documents: {mem0_collection.count()}")

    # 读取所有文档正文和 metadata。
    raw_docs = mem0_collection.get(include=["documents", "metadatas"])

    # 准备一个列表，专门保存 Sarah 的底层记忆文档。
    sarah_docs = []
    # 同时遍历 ID、文档正文和 metadata。
    for doc_id, doc, meta in zip(
        # 取出所有文档 ID。
        raw_docs.get("ids", []),
        # 取出所有文档正文。
        raw_docs.get("documents", []),
        # 取出所有 metadata。
        raw_docs.get("metadatas", [])
    ):
        # 如果 metadata 属于 Sarah。
        if isinstance(meta, dict) and meta.get("user_id") == sarah_user_id:
            # 就把这条记录加入 Sarah 专属列表。
            sarah_docs.append((doc_id, doc, meta))

    # 打印 Sarah 当前命中的文档数。
    print(f"\nDocuments for {sarah_user_id}: {len(sarah_docs)}")
    # 最多展示前 10 条 Sarah 的底层文档。
    for doc_id, doc, meta in sarah_docs[:10]:
        # 打印文档 ID。
        print(f"\nDocument ID: {doc_id}")
        # 打印 metadata。
        print(f"  Metadata: {meta}")
        # 打印记忆正文。
        print(f"  Memory: {doc}")

# 如果读取底层集合失败。
except Exception as e:
    # 打印错误信息。
    print(f"Error checking collection: {str(e)}")



🔍 CHECKING MEM0 COLLECTION STRUCTURE

Collection name: mem0
Total documents: 19

Documents for sarah_johnson_123: 13

Document ID: 00b2132b-c338-4ebc-99e0-5342cb16c96e
  Metadata: {'attributed_to': 'user', 'created_at': '2026-05-03T15:40:32.212646+00:00', 'data': 'User is planning a 10th wedding anniversary trip with a focus on romantic destinations, fine dining, and spa experiences.', 'hash': '021d0ad9dd358cb5582cd5dcf2daae9a', 'text_lemmatized': 'user plan planning 10th wedding anniversary trip focus romantic destination fine dining spa experience', 'updated_at': '2026-05-03T15:40:32.212646+00:00', 'user_id': 'sarah_johnson_123'}
  Memory: None

Document ID: febe2b28-ebc6-480f-81a1-ce828c7cdbb4
  Metadata: {'attributed_to': 'user', 'created_at': '2026-05-03T15:40:32.214750+00:00', 'data': 'User is looking for accessible accommodations for their 10th wedding anniversary trip.', 'hash': '753b15b1aedf1663d02046e6f16f26a0', 'text_lemmatized': 'user look looking accessible accommodation 

## 演示语义记忆搜索

Mem0的强大之处在于语义搜索——根据含义而不仅仅是关键词来查找相关记忆。


In [50]:
# 打印标题，开始演示 Mem0 的语义记忆搜索能力。
print("🔍 SEMANTIC MEMORY SEARCH DEMONSTRATION\n")

# 用一个和“饮食限制”语义相关的查询来搜索 Sarah 的记忆。
dietary_search = memory.search(
    # 查询文本会让模型尽量召回饮食、食物过敏、限制相关的记忆。
    "dietary food allergies restrictions", filters={"user_id": sarah_user_id})

# 如果返回值是带 results 的字典结构。
if isinstance(dietary_search, dict) and 'results' in dietary_search:
    # 就取出真正的结果列表。
    dietary_results = dietary_search.get('results', [])
# 否则如果它本身已经是列表。
else:
    # 就直接使用它；如果两者都不是，则回退为空列表。
    dietary_results = dietary_search if isinstance(
        dietary_search, list) else []

# 打印本次搜索的查询词。
print("Search Query: 'dietary food allergies restrictions'")
# 打印结果所属用户。
print(f"Results for Sarah:")
# 打印一条分隔线，便于阅读。
print("=" * 50)
# 如果确实查到了结果。
if dietary_results:
    # 遍历每一条结果。
    for mem in dietary_results:
        # 如果结果是字典格式。
        if isinstance(mem, dict):
            # 打印记忆正文。
            print(f"- {mem.get('memory', 'Unknown')}")
            # 打印相关性分数。
            print(f"  Relevance Score: {mem.get('score', 'N/A')}")
        # 如果结果不是字典。
        else:
            # 就直接打印它的字符串内容。
            print(f"- {mem}")
# 如果没有查到结果。
else:
    # 打印未命中提示。
    print("- No memories found")


Number of requested results 80 is greater than number of elements in index 19, updating n_results = 19


🔍 SEMANTIC MEMORY SEARCH DEMONSTRATION

Search Query: 'dietary food allergies restrictions'
Results for Sarah:
- budget is around $700-800 per night for the trip
  Relevance Score: 1.0
- budget is around $700-800 per night for the trip
  Relevance Score: 1.0
- romantic destinations, fine dining, and spa experiences for 10th wedding anniversary
  Relevance Score: 1.0
- romantic destinations, fine dining, and spa experiences for 10th wedding anniversary
  Relevance Score: 1.0
- User is planning a 10th wedding anniversary trip with a focus on romantic destinations, fine dining, and spa experiences.
  Relevance Score: 1.0
- User has a budget of $700-800 per night for their 10th wedding anniversary trip.
  Relevance Score: 1.0
- I love romantic destinations with spa services
  Relevance Score: 1.0
- User's husband has mobility issues, which is why they need accessible accommodations for their 10th wedding anniversary trip
  Relevance Score: 0.9879030543723853
- User is looking for accessibl

## 关键要点

### 1. 持久的用户记忆
- **跨会话持久性**：用户偏好在不同对话中得以保留
- **用户隔离**：每个用户都有自己的记忆空间
- **自动上下文**：代理会自动检索相关记忆

### 2. Mem0 的优势
- **语义理解**：不仅记住关键词，还能理解含义
- **自动提取**：从对话中自动提取值得记忆的信息
- **相关性排序**：优先返回最相关的历史记忆

### 3. ChromaDB 的作用
- **本地可运行**：不依赖 Azure AI Search
- **向量检索**：支持酒店搜索和记忆召回
- **适合学习**：便于本地实验和调试

### 4. 生产环境思路
- 学习阶段可以使用本地 `ChromaDB`
- 上线阶段可以替换为托管向量数据库或搜索服务
- `Mem0 + Semantic Kernel` 的整体设计不需要改变


## 摘要

恭喜你！你已经成功构建了一个具有持久记忆功能的 AI 旅行助手，使用了以下技术：

- **Mem0**：用于智能的持久记忆管理
- **ChromaDB**：作为本地的酒店检索和记忆向量存储
- **Semantic Kernel**：用于构建工具可调用的智能代理
- **Qwen / DashScope OpenAI Compatible API**：作为聊天模型
- **本地 BGE 小模型**：作为 embedding 服务，为记忆和检索生成向量

你在本 notebook 中学到了：

1. 如何将 Mem0 与本地 ChromaDB 集成以实现持久记忆
2. 如何通过 OpenAI 兼容接口把 `qwen-max` 接入 Semantic Kernel
3. 如何把项目本地 `BAAI/bge-small-en-v1.5` 用作 embedding 模型
4. 如何在多轮和跨会话场景中复用用户记忆
5. 如何在不依赖 Azure AI Search 和远程 embedding 服务的情况下完成同样的教学实验



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保准确性，但请注意，自动翻译可能包含错误或不准确之处。应以原始语言的文档为权威来源。对于关键信息，建议使用专业人工翻译。因使用本翻译而引起的任何误解或误读，我们概不负责。
